# 第2课：自动微分（Autograd）

**学习目标：**
- 理解 `requires_grad` 的作用
- 掌握 `loss.backward()` 的梯度计算
- 与 NumPy 教程中手写的 `Value` 类对比

---

在 NumPy 教程的第8课，我们手动实现了 `Value` 类来构建计算图和自动微分。PyTorch 的 `autograd` 做的是同样的事情，但更高效、更完善。

核心概念：**设置了 `requires_grad=True` 的 Tensor，PyTorch 会自动跟踪所有运算，调用 `.backward()` 时自动计算梯度。**

## 2.1 基本用法

创建一个需要梯度的 Tensor，进行运算后调用 `backward()`：

In [ ]:
import torch

# 创建需要梯度的 Tensor
x = torch.tensor(2.0, requires_grad=True)
print("x:", x)
print("requires_grad:", x.requires_grad)  # True

# 运算：y = x^2 + 3x + 1
y = x**2 + 3*x + 1
print("y:", y)  # 2^2 + 3*2 + 1 = 11

# 反向传播：计算 dy/dx
y.backward()

# dy/dx = 2x + 3 = 2*2 + 3 = 7
print("dy/dx:", x.grad)  # tensor(7.)

## 2.2 向量的梯度

当输入是向量时，`backward()` 需要一个与输出同形状的 `gradient` 参数（通常是全1向量）。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# y = x^2
y = x ** 2

# 对向量求梯度需要传入 gradient 参数
y.backward(gradient=torch.ones_like(x))

# dy/dx = 2x = [2, 4, 6]
print("dy/dx:", x.grad)  # tensor([2., 4., 6.])

## 2.3 矩阵运算的梯度

神经网络中最常见的运算：矩阵乘法 + 损失函数。

In [ ]:
# 输入
X = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])  # (2, 2)

# 权重（需要梯度）
W = torch.randn(2, 2, requires_grad=True)

# 前向传播
Z = X @ W          # (2, 2)
loss = Z.sum()      # 简单的损失：所有元素求和

print("loss:", loss.item())

# 反向传播
loss.backward()

# W 的梯度
print("W 的梯度:\n", W.grad)
print("梯度形状:", W.grad.shape)  # (2, 2)，与 W 同形状

## 2.4 梯度清零

**重要：** PyTorch 默认会累积梯度，每次反向传播前必须手动清零。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

# 第一次
loss1 = (w * 2) ** 2
loss1.backward()
print("第1次梯度:", w.grad)  # 8.0

# 第二次（不清零会累积！）
loss2 = (w * 3) ** 2
loss2.backward()
print("累积梯度:", w.grad)  # 8 + 18 = 26（错误！应该是18）

# 正确做法：手动清零
w.grad.zero_()
loss2.backward()
print("清零后:", w.grad)  # 18.0

## 2.5 关闭梯度追踪

推理（测试）时不需要计算梯度，用 `torch.no_grad()` 可以节省内存和计算：

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# 需要梯度
y = x ** 2
y.backward()
print("有梯度:", x.grad)  # 4.0

# 不需要梯度
with torch.no_grad():
    z = x ** 3
    print("z:", z)
    print("z.requires_grad:", z.requires_grad)  # False

---

## 与手写 Value 类的对比

| 特性 | 手写 Value 类 | PyTorch autograd |
| --- | --- | --- |
| 计算图 | 手动构建 | 自动构建 |
| 梯度计算 | 标量级，逐个运算 | 向量化，GPU 加速 |
| backward() | 拓扑排序 + 链式法则 | 同样的原理，但高度优化 |
| 用途 | 学习原理 | 生产环境 |

---

## 小结

- `requires_grad=True` 让 PyTorch 自动追踪运算
- `.backward()` 自动计算所有梯度
- **训练前必须 `.grad.zero_()` 清零**
- `torch.no_grad()` 用于推理阶段

**下一课**我们将用 `nn.Module` 构建正式的神经网络。